In [ ]:
# %%
# Import necessary libraries
import matplotlib.pyplot as plt
import polars as pl
import numpy as np
import pandas as pd # Added for Plotly compatibility
import seaborn as sns
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, ExpSineSquared,
    DotProduct, WhiteKernel, ConstantKernel as C
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    r2_score, mean_squared_error, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
)
import optuna

# Import Plotly libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# %%
# --- 1. Data Loading and Preprocessing ---

# Load the heart disease dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'DISEASE']

raw_data = pl.read_csv(
    url,
    infer_schema_length=1000,
    has_header=False,
    new_columns=columns,
    null_values='?' # Directly handle '?' as null
)

# Binarize the target variable: 0 = no disease, >0 = has disease
# This simplifies the problem to binary classification.
raw_data = (
    raw_data
    .with_columns(
        DISEASE=pl.when(pl.col('DISEASE') > 0)
        .then(1)
        .otherwise(0)
        .alias('DISEASE') # Ensure the column is renamed correctly
    )
)

# Clean and cast data types
raw_data = (
    raw_data
    .with_columns([
        pl.col('age').cast(pl.Int8),
        pl.col('ca').cast(pl.Float64),
        pl.col('thal').cast(pl.Float64),
        pl.col('sex').cast(pl.Int8),
        pl.col('cp').cast(pl.Int8),
        pl.col('DISEASE').cast(pl.Int8),
        pl.col('fbs').cast(pl.Int8),
        pl.col('restecg').cast(pl.Int8),
        pl.col('exang').cast(pl.Int8),
        pl.col('trestbps').cast(pl.Float64),
        pl.col('chol').cast(pl.Float64),
        pl.col('thalach').cast(pl.Float64),
        pl.col('oldpeak').cast(pl.Float64),
        pl.col('slope').cast(pl.Int8),
    ])
)

# Fill missing values with the mean of their respective columns
raw_data = (
    raw_data
    .with_columns([
        pl.col('ca').fill_null(pl.col('ca').mean()),
        pl.col('thal').fill_null(pl.col('thal').mean()),
    ])
)

print("Data loaded and preprocessed. Head of the dataframe:")
print(raw_data.head())
print("\nDisease distribution:")
print(raw_data['DISEASE'].value_counts().sort('DISEASE'))


# %%
# --- 2. Feature and Label Preparation ---

# Select features and label
features = raw_data.select(pl.selectors.exclude('DISEASE'))
label = raw_data['DISEASE']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    features.to_numpy(), # Convert to numpy for sklearn compatibility
    label.to_numpy(),
    test_size=0.2,
    random_state=42,
    stratify=label # Ensure balanced split for binary classification
)

# Normalize the data using z-score distribution
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# %%
# --- 3. Optuna Kernel Optimization ---

def create_kernel(trial):
    """
    Create a Gaussian Process kernel based on Optuna trial suggestions.
    Explores various kernel types and their combinations.
    """
    kernel_type = trial.suggest_categorical(
        'kernel_type',
        ['RBF', 'Matern', 'RationalQuadratic', 'RBF+Matern',
         'RBF+RationalQuadratic', 'Matern+RationalQuadratic',
         'RBF*Matern', 'Product_All']
    )

    # Common parameters
    constant_value = trial.suggest_float('constant_value', 0.1, 10.0, log=True)
    alpha = trial.suggest_float('alpha', 1e-5, 1e-1, log=True)

    # More reasonable bounds for length scales
    length_scale_bounds = (1e-2, 1e2)

    if kernel_type == 'RBF':
        length_scale_rbf = trial.suggest_float('length_scale_rbf', 0.1, 10.0)
        kernel = C(constant_value, (1e-3, 1e3)) * RBF(
            length_scale=length_scale_rbf, length_scale_bounds=length_scale_bounds
        )
    elif kernel_type == 'Matern':
        length_scale_matern = trial.suggest_float('length_scale_matern', 0.1, 10.0)
        nu = trial.suggest_categorical('nu', [0.5, 1.5, 2.5])
        kernel = C(constant_value, (1e-3, 1e3)) * Matern(
            length_scale=length_scale_matern, length_scale_bounds=length_scale_bounds, nu=nu
        )
    elif kernel_type == 'RationalQuadratic':
        length_scale_rq = trial.suggest_float('length_scale_rq', 0.1, 10.0)
        alpha_rq = trial.suggest_float('alpha_rq', 0.1, 10.0)
        kernel = C(constant_value, (1e-3, 1e3)) * RationalQuadratic(
            length_scale=length_scale_rq, alpha=alpha_rq,
            length_scale_bounds=length_scale_bounds, alpha_bounds=(1e-2, 1e2)
        )
    # ... (other kernel combinations follow the same pattern)
    elif kernel_type == 'RBF+Matern':
        length_scale_rbf = trial.suggest_float('length_scale_rbf', 0.1, 10.0)
        length_scale_matern = trial.suggest_float('length_scale_matern', 0.1, 10.0)
        nu = trial.suggest_categorical('nu', [0.5, 1.5, 2.5])
        kernel = C(constant_value, (1e-3, 1e3)) * (
            RBF(length_scale=length_scale_rbf, length_scale_bounds=length_scale_bounds) +
            Matern(length_scale=length_scale_matern, length_scale_bounds=length_scale_bounds, nu=nu)
        )
    elif kernel_type == 'RBF+RationalQuadratic':
        length_scale_rbf = trial.suggest_float('length_scale_rbf', 0.1, 10.0)
        length_scale_rq = trial.suggest_float('length_scale_rq', 0.1, 10.0)
        alpha_rq = trial.suggest_float('alpha_rq', 0.1, 10.0)
        kernel = C(constant_value, (1e-3, 1e3)) * (
            RBF(length_scale=length_scale_rbf, length_scale_bounds=length_scale_bounds) +
            RationalQuadratic(length_scale=length_scale_rq, alpha=alpha_rq,
                            length_scale_bounds=length_scale_bounds, alpha_bounds=(1e-2, 1e2))
        )
    elif kernel_type == 'Matern+RationalQuadratic':
        length_scale_matern = trial.suggest_float('length_scale_matern', 0.1, 10.0)
        nu = trial.suggest_categorical('nu', [0.5, 1.5, 2.5])
        length_scale_rq = trial.suggest_float('length_scale_rq', 0.1, 10.0)
        alpha_rq = trial.suggest_float('alpha_rq', 0.1, 10.0)
        kernel = C(constant_value, (1e-3, 1e3)) * (
            Matern(length_scale=length_scale_matern, length_scale_bounds=length_scale_bounds, nu=nu) +
            RationalQuadratic(length_scale=length_scale_rq, alpha=alpha_rq,
                            length_scale_bounds=length_scale_bounds, alpha_bounds=(1e-2, 1e2))
        )
    elif kernel_type == 'RBF*Matern':
        length_scale_rbf = trial.suggest_float('length_scale_rbf', 0.1, 10.0)
        length_scale_matern = trial.suggest_float('length_scale_matern', 0.1, 10.0)
        nu = trial.suggest_categorical('nu', [0.5, 1.5, 2.5])
        kernel = C(constant_value, (1e-3, 1e3)) * (
            RBF(length_scale=length_scale_rbf, length_scale_bounds=length_scale_bounds) *
            Matern(length_scale=length_scale_matern, length_scale_bounds=length_scale_bounds, nu=nu)
        )
    elif kernel_type == 'Product_All':
        length_scale_rbf = trial.suggest_float('length_scale_rbf', 0.1, 10.0)
        length_scale_matern = trial.suggest_float('length_scale_matern', 0.1, 10.0)
        nu = trial.suggest_categorical('nu', [0.5, 1.5, 2.5])
        length_scale_rq = trial.suggest_float('length_scale_rq', 0.1, 10.0)
        alpha_rq = trial.suggest_float('alpha_rq', 0.1, 10.0)
        kernel = C(constant_value, (1e-3, 1e3)) * (
            RBF(length_scale=length_scale_rbf, length_scale_bounds=length_scale_bounds) *
            Matern(length_scale=length_scale_matern, length_scale_bounds=length_scale_bounds, nu=nu) *
            RationalQuadratic(length_scale=length_scale_rq, alpha=alpha_rq,
                            length_scale_bounds=length_scale_bounds, alpha_bounds=(1e-2, 1e2))
        )
    return kernel, alpha


def objective(trial):
    """
    Optuna objective function to optimize GP with multiple kernel types using cross-validation.
    """
    try:
        kernel, alpha = create_kernel(trial)

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=alpha,
            n_restarts_optimizer=3,
            random_state=42,
            normalize_y=True
        )

        # Use 5-fold cross-validation for a more robust performance estimate
        # We optimize for R² score
        scores = cross_val_score(gp, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
        mean_r2 = np.mean(scores)

        # Store additional metrics for later analysis
        trial.set_user_attr('cv_r2_mean', mean_r2)
        trial.set_user_attr('cv_r2_std', np.std(scores))

        return mean_r2

    except Exception as e:
        # Return poor score if kernel fails to converge or fit
        print(f"Trial {trial.number} failed with kernel '{trial.params.get('kernel_type', 'N/A')}': {str(e)}")
        return -1.0

# %%
# --- 4. Run Optimization ---

print("\nStarting optimization with multiple kernel types...")
study = optuna.create_study(
    direction='maximize',
    study_name='multi_kernel_gp_optimization_binary',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# You can increase n_trials for a more thorough search
study.optimize(objective, n_trials=100, show_progress_bar=True)

# %%
# --- 5. Report Optimization Results ---

print("\n" + "="*60)
print("OPTIMIZATION RESULTS")
print("="*60)
print(f"Best trial number: {study.best_trial.number}")
print(f"Best CV R² Score: {study.best_trial.value:.4f} (±{study.best_trial.user_attrs['cv_r2_std']:.4f})")

print("\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

print("\nTOP 5 KERNEL CONFIGURATIONS")
print("="*60)
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -np.inf, reverse=True)
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"\nRank {i}:")
    print(f"  Kernel Type: {trial.params.get('kernel_type', 'N/A')}")
    print(f"  CV R² Score: {trial.value:.4f} (±{trial.user_attrs.get('cv_r2_std', 0):.4f})")


# %%
# --- 6. Final Model Training and Evaluation ---

print("\n" + "="*60)
print("FINAL MODEL TRAINING & EVALUATION")
print("="*60)

# Train the final model on the entire training set with the best kernel
best_kernel, best_alpha = create_kernel(study.best_trial)
final_gp = GaussianProcessRegressor(
    kernel=best_kernel,
    alpha=best_alpha,
    n_restarts_optimizer=10, # More restarts for final model
    random_state=42,
    normalize_y=True
)
final_gp.fit(X_train, y_train)

# Make predictions
y_pred_train = final_gp.predict(X_train)
y_pred_test, y_std = final_gp.predict(X_test, return_std=True)

# Convert regression predictions to binary labels for classification metrics
# Using a 0.5 threshold
y_pred_test_binary = (y_pred_test > 0.5).astype(int)
y_pred_train_binary = (y_pred_train > 0.5).astype(int)

# --- Regression Metrics ---
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("\n--- Regression Performance ---")
print(f"  Train R²: {r2_train:.4f}")
print(f"  Test R²:  {r2_test:.4f}")
print(f"  Test RMSE: {rmse_test:.4f}")
print(f"  Mean Prediction Std: {y_std.mean():.4f}")

# --- Classification Metrics ---
accuracy = accuracy_score(y_test, y_pred_test_binary)
precision = precision_score(y_test, y_pred_test_binary)
recall = recall_score(y_test, y_pred_test_binary)
f1 = f1_score(y_test, y_pred_test_binary)
auc = roc_auc_score(y_test, y_pred_test) # Use raw probabilities for AUC

print("\n--- Classification Performance (Threshold=0.5) ---")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  AUC:       {auc:.4f}")

print(f"\nOptimized Kernel:\n{final_gp.kernel_}")


# %%
# --- 7. Interactive Visualization with Plotly ---

# Create subplots
fig = make_subplots(
    rows=3, cols=4,
    subplot_titles=(
        'Optimization History', 'Kernel Type Performance', 'Predicted vs Actual', 'Residuals',
        'Confusion Matrix', 'ROC Curve', 'Predictions with Uncertainty', 'Uncertainty vs Error',
        'Best Score Progress', 'Hyperparameter Importance', '', ''
    ),
    specs=[[{"secondary_y": False}, {"secondary_y": False}, {"secondary_y": False}, {"secondary_y": False}],
           [{"type": "heatmap"}, {"secondary_y": False}, {"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}, {"type": "scatter"}, {"type": "scatter"}]]
)

# Plot 1: Optimization History
fig.add_trace(
    go.Scatter(x=trial_numbers, y=trial_values, mode='lines+markers', name='CV R² Score'),
    row=1, col=1
)
fig.add_hline(y=study.best_value, line_dash="dash", line_color="red", row=1, col=1)

# Plot 2: Kernel Type Performance
df_kernel_perf = pd.DataFrame(list(kernel_performance.items()), columns=['Kernel', 'Scores'])
df_kernel_perf['Mean'] = df_kernel_perf['Scores'].apply(np.mean)
df_kernel_perf['Std'] = df_kernel_perf['Scores'].apply(np.std)
fig.add_trace(
    go.Bar(x=df_kernel_perf['Mean'], y=df_kernel_perf['Kernel'], orientation='h', error_x=dict(type='data', array=df_kernel_perf['Std'])),
    row=1, col=2
)

# Plot 3: Predicted vs Actual (Test)
fig.add_trace(
    go.Scatter(x=y_test, y=y_pred_test, mode='markers', name='Test Predictions'),
    row=1, col=3
)
fig.add_shape(type="line", x0=y_test.min(), y0=y_test.min(), x1=y_test.max(), y1=y_test.max(), line=dict(color="red", dash="dash"), row=1, col=3)

# Plot 4: Residuals
fig.add_trace(
    go.Scatter(x=y_pred_test, y=residuals, mode='markers', name='Residuals'),
    row=1, col=4
)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=4)

# Plot 5: Confusion Matrix
fig.add_trace(
    go.Heatmap(z=cm, x=['Pred 0', 'Pred 1'], y=['True 0', 'True 1'], text=cm, texttemplate="%{text}", textfont={"color": "white"}, colorscale='Blues'),
    row=2, col=1
)

# Plot 6: ROC Curve
fig.add_trace(
    go.Scatter(x=fpr, y=tpr, name=f'ROC Curve (AUC = {auc:.2f})'),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Chance Level', line=dict(color='navy', dash='dash')),
    row=2, col=2
)

# Plot 7: Prediction Uncertainty
fig.add_trace(
    go.Scatter(x=np.arange(len(y_test)), y=y_test, error_y=dict(type='data', array=y_std*2), mode='markers', name='Actual ± 2σ'),
    row=2, col=3
)
fig.add_trace(
    go.Scatter(x=np.arange(len(y_test)), y=y_pred_test, mode='markers', name='Predicted', marker=dict(color='red')),
    row=2, col=3
)

# Plot 8: Uncertainty vs Error
fig.add_trace(
    go.Scatter(x=y_std, y=abs_errors, mode='markers', name='Error vs StdDev'),
    row=2, col=4
)

# Plot 9: Best Score Progress
best_values = [max([t.value for t in study.trials[:i+1] if t.value is not None]) for i in range(len(study.trials))]
fig.add_trace(
    go.Scatter(x=np.arange(len(best_values)), y=best_values, mode='lines', name='Best Score So Far', line=dict(color='green')),
    row=3, col=1
)

# Plot 10: Hyperparameter Importance
try:
    importance = optuna.importance.get_param_importances(study)
    df_importance = pd.DataFrame(list(importance.items()), columns=['param', 'importance']).sort_values('importance', ascending=True)
    fig.add_trace(
        go.Bar(x=df_importance['importance'], y=df_importance['param'], orientation='h', name='Importance'),
        row=3, col=2
    )
except Exception as e:
    fig.add_annotation(text=f"Importance calculation failed: {e}", x=0.5, y=0.5, xref="x domain", yref="y domain", showarrow=False, row=3, col=2)


# Update layout
fig.update_layout(
    height=1200,
    width=1600,
    title_text="Gaussian Process Model Analysis",
    showlegend=False # Hide individual legends to avoid clutter
)

# Show the interactive plot
fig.show()

# Save the interactive plot as an HTML file
fig.write_html("gp_model_results.html")
print("\nInteractive plots saved as 'gp_model_results.html'")

# To save as a static image (e.g., PNG), you need to install 'kaleido':
# pip install kaleido
# fig.write_image("gp_model_results.png")